# GOL651 – 03 · Optimisation avec Pyomo (HiGHS, CBC, GLPK)
*GOL651 – 03 · Optimization with Pyomo*

**Objectifs.** Formuler et résoudre un programme linéaire (plan de production), interpréter les prix duaux, puis résoudre un programme linéaire en nombres entiers (localisation d'entrepôts) avec trois solveurs.

*Goals: formulate and solve an LP (production plan), interpret dual prices, then solve a MILP (warehouse location) with three solvers.*

In [ ]:
import pyomo
import pyomo.environ as pyo
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
print("Pyomo", pyomo.__version__)

In [ ]:
from pathlib import Path

def find_project_root(start: Path = Path.cwd()) -> Path:
    """Folder that contains data/datasets (works inside Docker and on a laptop)."""
    for folder in [start, *start.parents]:
        if (folder / "data" / "datasets").is_dir():
            return folder
    raise FileNotFoundError("Project root not found: open this notebook from the course folder")

ROOT = find_project_root()
DATA = ROOT / "data"
print("Project folder:", ROOT)

Solveurs disponibles / *Available solvers*

In [ ]:
pd.DataFrame(
    [(name, pyo.SolverFactory(name).available(exception_flag=False)) for name in ("highs", "cbc", "glpk")],
    columns=["solver", "available"],
)

## 1. Programme linéaire : plan de production / *LP: production plan*

Un atelier de Laval fabrique trois produits en bois. Variables : $x_j \ge 0$, unités du produit $j$ fabriquées cette semaine.

$$
\begin{aligned}
\max \quad & \sum_{j \in P} p_j\, x_j \\
\text{s.c.} \quad & \sum_{j \in P} a_{ij}\, x_j \le b_i && \forall i \in R \quad \text{(ressources)} \\
& 0 \le x_j \le d_j && \forall j \in P \quad \text{(demande)}
\end{aligned}
$$

*Maximize profit subject to resource capacities and demand limits.*

In [ ]:
P = ["standard_pallet", "reinforced_pallet", "shipping_crate"]
R = ["wood_m3", "assembly_h", "finishing_h"]
profit   = {"standard_pallet": 12.0, "reinforced_pallet": 19.0, "shipping_crate": 30.0}
demand   = {"standard_pallet": 400,  "reinforced_pallet": 250,  "shipping_crate": 120}
capacity = {"wood_m3": 60.0, "assembly_h": 150.0, "finishing_h": 70.0}
usage = {
    ("wood_m3", "standard_pallet"): 0.06, ("wood_m3", "reinforced_pallet"): 0.09, ("wood_m3", "shipping_crate"): 0.15,
    ("assembly_h", "standard_pallet"): 0.15, ("assembly_h", "reinforced_pallet"): 0.25, ("assembly_h", "shipping_crate"): 0.40,
    ("finishing_h", "standard_pallet"): 0.05, ("finishing_h", "reinforced_pallet"): 0.10, ("finishing_h", "shipping_crate"): 0.25,
}

def build_lp(assembly_hours: float = 150.0) -> pyo.ConcreteModel:
    m = pyo.ConcreteModel()
    m.P = pyo.Set(initialize=P)
    m.R = pyo.Set(initialize=R)
    m.x = pyo.Var(m.P, bounds=lambda m, j: (0, demand[j]))
    cap = {**capacity, "assembly_h": assembly_hours}
    m.profit = pyo.Objective(expr=sum(profit[j] * m.x[j] for j in m.P), sense=pyo.maximize)
    m.resource = pyo.Constraint(m.R, rule=lambda m, i: sum(usage[i, j] * m.x[j] for j in m.P) <= cap[i])
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m

lp = build_lp()
result = pyo.SolverFactory("highs").solve(lp)
assert pyo.check_optimal_termination(result)

print(f"Profit: {pyo.value(lp.profit):,.2f} $")
pd.DataFrame({"units": {j: pyo.value(lp.x[j]) for j in P}, "demand_max": demand}).round(1)

### Prix duaux (prix implicites) / *Dual (shadow) prices*
Le prix dual d'une ressource est le profit supplémentaire obtenu avec **une unité de plus** de cette ressource. Une ressource non saturée a un prix dual nul.

*The dual price is the extra profit from one more unit of a resource; a non-binding resource has a zero dual price.*

In [ ]:
pd.DataFrame({
    "used": {i: pyo.value(sum(usage[i, j] * lp.x[j] for j in P)) for i in R},
    "available": capacity,
    "dual_price_$": {i: lp.dual[lp.resource[i]] for i in R},
}).round(2)

### Analyse paramétrique / *Parametric analysis*
Que rapporte chaque heure d'assemblage supplémentaire ? On résout le modèle pour plusieurs capacités.

*How much is each extra assembly hour worth? We re-solve the model for several capacities.*

In [ ]:
hours = list(range(110, 211, 10))
profits = []
for h in hours:
    m = build_lp(h)
    pyo.SolverFactory("highs").solve(m)
    profits.append(pyo.value(m.profit))

plt.figure(figsize=(7, 3.5))
plt.plot(hours, profits, marker="o")
plt.title("Optimal profit vs assembly capacity")
plt.xlabel("Assembly hours available")
plt.ylabel("Profit ($)")
plt.grid(alpha=0.3)
plt.show()

## 2. Programme en nombres entiers : localisation d'entrepôts / *MILP: warehouse location*

Variables binaires $y_i \in \{0,1\}$ (ouvrir l'entrepôt $i$) et flux $x_{ij} \ge 0$ :

$$
\begin{aligned}
\min \quad & \sum_{i} f_i\, y_i + \sum_{i}\sum_{j} c_{ij}\, x_{ij} \\
\text{s.c.} \quad & \sum_{i} x_{ij} = d_j && \forall j \\
& \sum_{j} x_{ij} \le u_i\, y_i && \forall i \\
& x_{ij} \le d_j\, y_i && \forall i, j
\end{aligned}
$$

Le modèle est défini dans `examples/optimization/facility_location_milp.py` ; on le réutilise ici.

*The model lives in a Python module and is reused here.*

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "examples" / "optimization"))
import facility_location_milp as flp

rows = []
for name in ("highs", "cbc", "glpk"):
    m = flp.solve(name)
    opened = [i for i in m.I if pyo.value(m.y[i]) > 0.5]
    rows.append((name, ", ".join(opened), round(pyo.value(m.total_cost))))
pd.DataFrame(rows, columns=["solver", "opened", "total_cost_$"])

### Pourquoi des variables entières ? / *Why integer variables?*
En relâchant $y_i \in \{0,1\}$ en $0 \le y_i \le 1$, le modèle devient un programme linéaire : il « ouvre à moitié » des entrepôts et sous-estime le coût. L'écart est le **saut d'intégrité**.

*Relaxing the binaries gives an LP that "half-opens" warehouses and underestimates the cost: the integrality gap.*

In [ ]:
relaxed = flp.build_model()
pyo.TransformationFactory("core.relax_integer_vars").apply_to(relaxed)
pyo.SolverFactory("highs").solve(relaxed)

milp_cost = pyo.value(flp.solve("highs").total_cost)
lp_cost = pyo.value(relaxed.total_cost)
print(f"MILP cost: {milp_cost:,.0f} $   LP relaxation: {lp_cost:,.0f} $   gap: {100 * (milp_cost - lp_cost) / milp_cost:.1f} %")
pd.Series({i: round(pyo.value(relaxed.y[i]), 3) for i in relaxed.I}, name="y (relaxed)")

### À vous / *Your turn*
1. Augmentez de 50 % le coût fixe de Boucherville : la solution change-t-elle ? / *Increase Boucherville's fixed cost by 50 %: does the solution change?*
2. Ajoutez une contrainte « au plus un entrepôt sur la Rive-Sud ». / *Add a constraint "at most one South Shore warehouse".*